# VeritasRAG: does hybrid retrieval + re-ranking + answer guards reduce hallucination?

This notebook builds a small search-and-answer system and **measures** whether each part of it reduces hallucination.
It uses SQuAD 2.0, which has questions that **can** be answered from the text and questions that **cannot**.
A system that answers an unanswerable question is hallucinating, so we can count it automatically.

Five configurations are compared on the same held-out questions:

| Config | Retrieval | Re-rank | Guards |
|---|---|---|---|
| A | keyword (BM25) | no | no |
| B | dense embeddings | no | no |
| C | hybrid (BM25 + dense, rank fusion) | no | no |
| D | hybrid | cross-encoder | no |
| E | hybrid | cross-encoder | score threshold + grounding check |

Run the cells in order. Set `QUICK_TEST = True` first to check everything works (a few minutes), then `False` for the full run.
Everything is saved in `/content/veritasrag_out` (not Google Drive), and the last cell zips it for download.

In [ ]:
# Cell 1: install (about 1-2 minutes). Runtime > Change runtime type > T4 GPU first.
!pip -q install rank_bm25 sentence-transformers datasets transformers accelerate

In [ ]:
# Cell 2: settings
MOCK = False            # keep False in Colab. True is only used for offline testing of the logic.
QUICK_TEST = True       # True = tiny run to check the pipeline. Set False for the real run.

SEED = 7
N_ARTICLES = 60         # how many SQuAD articles form the search corpus
DEV_N, TEST_N = 100, 300  # dev tunes the guard threshold. test is only used for the final numbers. 50% unanswerable each.
RETRIEVE_K = 20         # candidates fetched before re-ranking
CONTEXT_K = 3           # passages given to the language model
GENERATOR = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDER = "BAAI/bge-small-en-v1.5"
RERANKER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
OUT = "/content/veritasrag_out"
if QUICK_TEST:
    N_ARTICLES, DEV_N, TEST_N = 15, 20, 40

In [ ]:
# Cell 3: helpers (text normalisation and metrics, same rules as the official SQuAD scoring)
import os, re, json, string, random, collections, time
import numpy as np, pandas as pd
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def norm(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1(pred, golds):
    best = 0.0
    p = norm(pred).split()
    for g in golds:
        g = norm(g).split()
        common = collections.Counter(p) & collections.Counter(g)
        n = sum(common.values())
        if n == 0: continue
        pr, rc = n / len(p), n / len(g)
        best = max(best, 2 * pr * rc / (pr + rc))
    return best

ABSTAIN_WORDS = ("unanswerable", "i don't know", "i do not know", "cannot be answered", "not in the context", "no answer")
def is_abstain(text):
    t = text.strip().lower()
    return t == "" or any(w in t for w in ABSTAIN_WORDS)

def tok(s): return re.findall(r"\w+", s.lower())

In [ ]:
# Cell 4: data. Passages form the search corpus. Questions are split into dev (tune) and test (report).
if MOCK:
    exec(open("/tmp/mock_data.py").read())
else:
    from datasets import load_dataset
    ds = load_dataset("rajpurkar/squad_v2", split="validation")
    titles = sorted(set(ds["title"]))
    random.Random(SEED).shuffle(titles)
    keep = set(titles[:N_ARTICLES])
    rows = [r for r in ds if r["title"] in keep]
    passages = sorted(set(r["context"] for r in rows))
    pid = {p: i for i, p in enumerate(passages)}
    questions = [dict(qid=r["id"], question=r["question"], golds=list(r["answers"]["text"]),
                      answerable=len(r["answers"]["text"]) > 0, gold_pid=pid[r["context"]]) for r in rows]

rng = random.Random(SEED)
ans = [q for q in questions if q["answerable"]]; una = [q for q in questions if not q["answerable"]]
rng.shuffle(ans); rng.shuffle(una)
h = lambda n: n // 2
dev = ans[:h(DEV_N)] + una[:h(DEV_N)]
test = ans[h(DEV_N):h(DEV_N) + h(TEST_N)] + una[h(DEV_N):h(DEV_N) + h(TEST_N)]
print(f"{len(passages)} passages | dev {len(dev)} | test {len(test)} (half unanswerable)")

In [ ]:
# Cell 5: retrievers (keyword, dense, hybrid) and the cross-encoder re-ranker
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([tok(p) for p in passages])

def bm25_rank(q, k):
    return list(np.argsort(-bm25.get_scores(tok(q)))[:k])

if MOCK:
    exec(open("/tmp/mock_models.py").read())
else:
    import torch
    from sentence_transformers import SentenceTransformer, CrossEncoder
    dev_dev = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = SentenceTransformer(EMBEDDER, device=dev_dev)
    P_emb = embedder.encode(passages, normalize_embeddings=True, batch_size=64, show_progress_bar=True)
    QPREFIX = "Represent this sentence for searching relevant passages: "
    def dense_rank(q, k):
        v = embedder.encode(QPREFIX + q, normalize_embeddings=True)
        return list(np.argsort(-(P_emb @ v))[:k])
    cross = CrossEncoder(RERANKER, device=dev_dev)
    def rerank(q, cand):
        s = cross.predict([(q, passages[i]) for i in cand], batch_size=32)
        order = np.argsort(-s)
        return [cand[i] for i in order], [float(s[i]) for i in order]

def rrf(lists, k=60):
    sc = collections.defaultdict(float)
    for lst in lists:
        for r, i in enumerate(lst): sc[i] += 1.0 / (k + r + 1)
    return [i for i, _ in sorted(sc.items(), key=lambda x: -x[1])]

def hybrid_rank(q, k):
    return rrf([bm25_rank(q, 50), dense_rank(q, 50)])[:k]

In [ ]:
# Cell 6: the language model that writes the answer
SYSTEM = ("Answer the question using only the context. Reply with a short answer copied from the context. "
          "If the context does not contain the answer, reply exactly: UNANSWERABLE")
if MOCK:
    pass  # generate() is defined in mock_models.py
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    gtok = AutoTokenizer.from_pretrained(GENERATOR)
    gmod = AutoModelForCausalLM.from_pretrained(GENERATOR, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(dev_dev)
    def generate(question, ctx_ids):
        ctx = "\n\n".join(passages[i] for i in ctx_ids)
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {question}"}]
        text = gtok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        enc = gtok(text, return_tensors="pt").to(dev_dev)
        with torch.no_grad():
            out = gmod.generate(**enc, max_new_tokens=24, do_sample=False, pad_token_id=gtok.eos_token_id)
        return gtok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# Cell 7: run configurations A to D on dev and test. Results are cached so a crash can resume.
CACHE = f"{OUT}/runs.jsonl"
done = {}
if os.path.exists(CACHE):
    for line in open(CACHE):
        r = json.loads(line); done[(r["cfg"], r["qid"])] = r

def run_one(cfg, q):
    t0 = time.time()
    score = None
    if cfg == "A": ids = bm25_rank(q["question"], CONTEXT_K)
    elif cfg == "B": ids = dense_rank(q["question"], CONTEXT_K)
    elif cfg == "C": ids = hybrid_rank(q["question"], CONTEXT_K)
    else:
        cand = hybrid_rank(q["question"], RETRIEVE_K)
        ids, scs = rerank(q["question"], cand); ids = ids[:CONTEXT_K]; score = scs[0]
    pred = generate(q["question"], ids)
    return dict(cfg=cfg, qid=q["qid"], ids=[int(i) for i in ids], score=score, pred=pred, secs=time.time() - t0)

with open(CACHE, "a") as fh:
    for split_name, split in (("dev", dev), ("test", test)):
        cfgs = ["D"] if split_name == "dev" else ["A", "B", "C", "D"]
        for cfg in cfgs:
            for q in split:
                if (cfg, q["qid"]) in done: continue
                r = run_one(cfg, q); r["split"] = split_name
                fh.write(json.dumps(r) + "\n"); fh.flush(); done[(cfg, q["qid"])] = r
            print("finished", split_name, cfg)

In [ ]:
# Cell 8: the guards (config E). Two checks, both automatic:
#   1. re-ranker score threshold: if the best passage scores too low, the system abstains without answering.
#   2. grounding check: if the answer text is not found in the retrieved passages, the system abstains.
# The threshold is chosen on DEV only, then applied unchanged to TEST.
qa = {q["qid"]: q for q in dev + test}

def grounded(pred, ids):
    ctx = norm(" ".join(passages[i] for i in ids))
    return norm(pred) != "" and norm(pred) in ctx

def outcome(q, pred, ids):
    # returns (answered, correct, grounded_ok)
    ab = is_abstain(pred)
    if q["answerable"]: corr = (not ab) and f1(pred, q["golds"]) >= 0.5
    else: corr = ab
    return (not ab), corr, (ab or grounded(pred, ids))

def guard(r, thr):
    # apply the guards to a config-D record. returns the final prediction
    if r["score"] is not None and r["score"] < thr: return "UNANSWERABLE"
    if not is_abstain(r["pred"]) and not grounded(r["pred"], r["ids"]): return "UNANSWERABLE"
    return r["pred"]

dev_recs = [done[("D", q["qid"])] for q in dev]
cands = sorted(set(round(r["score"], 3) for r in dev_recs)) + [-1e9]
def dev_acc(thr):
    return np.mean([outcome(qa[r["qid"]], guard(r, thr), r["ids"])[1] for r in dev_recs])
best_thr = max(cands, key=lambda t: (dev_acc(t), -abs(t)))
print(f"chosen threshold (from dev only): {best_thr:.3f} | dev accuracy {dev_acc(best_thr):.3f} vs no guard {dev_acc(-1e9):.3f}")

In [ ]:
# Cell 9: results on the held-out TEST questions, with 95% bootstrap confidence intervals over questions
def records(cfg):
    out = []
    for q in test:
        r = done[("D" if cfg == "E" else cfg, q["qid"])]
        pred = guard(r, best_thr) if cfg == "E" else r["pred"]
        ans_, corr, gr = outcome(q, pred, r["ids"])
        out.append(dict(qid=q["qid"], answerable=q["answerable"], answered=ans_, correct=corr, grounded=gr,
                        recall3=(q["gold_pid"] in r["ids"]), pred=pred))
    return pd.DataFrame(out)

def boot(x, n=2000):
    x = np.asarray(x, float); rs = np.random.RandomState(SEED)
    m = [x[rs.randint(0, len(x), len(x))].mean() for _ in range(n)]
    return float(x.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

rows = []
for cfg in "ABCDE":
    d = records(cfg); a, u = d[d.answerable], d[~d.answerable]
    row = dict(config=cfg)
    for name, series in (("recall@3 (answerable)", a.recall3), ("answered correctly (answerable)", a.correct),
                         ("hallucination rate (unanswerable)", u.answered),
                         ("unsupported answers (all)", d.answered & ~d.grounded), ("overall accuracy", d.correct)):
        m, lo, hi = boot(series); row[name] = f"{m:.3f} [{lo:.3f}, {hi:.3f}]"
    rows.append(row)
res = pd.DataFrame(rows).set_index("config")
res.to_csv(f"{OUT}/results.csv")
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
res

In [ ]:
# Cell 10: one picture of the trade-off. Lower hallucination is good, but not at the cost of answering fewer real questions.
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
pts = []
for cfg in "ABCDE":
    d = records(cfg)
    pts.append((cfg, d[~d.answerable].answered.mean(), d[d.answerable].correct.mean()))
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for cfg, hx, cy in pts:
    ax.scatter(hx, cy, s=70); ax.annotate(cfg, (hx, cy), xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("hallucination rate on unanswerable questions (lower is better)")
ax.set_ylabel("answered correctly on answerable questions (higher is better)")
ax.set_title(f"VeritasRAG ablation (n={len(test)} test questions)")
ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(f"{OUT}/tradeoff.png", dpi=160); plt.show()

In [ ]:
# Cell 11: read some failures yourself (this is the honest part of any evaluation)
d = records("E"); d["question"] = [qa[i]["question"] for i in d.qid]; d["gold"] = [qa[i]["golds"][:1] for i in d.qid]
bad = d[~d.correct].sample(min(15, int((~d.correct).sum())), random_state=SEED)
bad[["question", "answerable", "gold", "pred"]].to_csv(f"{OUT}/failures_to_read.csv", index=False)
bad[["question", "answerable", "gold", "pred"]]

In [ ]:
# Cell 12: zip everything for download
!cd /content && zip -qr veritasrag_out.zip veritasrag_out
from google.colab import files
files.download("/content/veritasrag_out.zip")

## How to read the results (write this part yourself)
- Compare A, B, C: does hybrid beat each alone on recall@3?
- Compare C and D: how much does the cross-encoder add?
- Compare D and E: do the guards cut the hallucination rate, and how many real answers did they cost?
- Quote the confidence intervals. If two intervals overlap heavily, say the difference is not clear.
- Mention the limits: one dataset (Wikipedia text), a 1.5B generator, a small test set, automatic metrics that are only a proxy for hallucination.